In [0]:
storageAccountName = ""
storageAccountAccessKey = ""
sasToken = ""

def mount_adls(blobContainerName):
    try:
      dbutils.fs.mount(
        source = "wasbs://{}@{}.blob.core.windows.net".format(blobContainerName, storageAccountName),
        mount_point = f"/mnt/{storageAccountName}/{blobContainerName}",
        #extra_configs = {'fs.azure.account.key.' + storageAccountName + '.blob.core.windows.net': storageAccountAccessKey}
        extra_configs = {'fs.azure.sas.' + blobContainerName + '.' + storageAccountName + '.blob.core.windows.net': sasToken}
      )
      print("OK!")
    except Exception as e:
      print("Falha", e)

In [0]:
mount_adls('gold')

In [0]:
df_assistencias  = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/assistencias")
df_atores        = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/atores")
df_avaliacoes    = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/avaliacoes")
df_episodios     = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/episodios")
df_filmes        = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/filmes")
df_generos       = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/generos")
df_pagamentos    = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/pagamentos")
df_planos        = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/planos")
df_series        = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/series")
df_usuarios      = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/usuarios")

In [ ]:
df_assistencias.createOrReplaceTempView("assistencias_silver")
df_atores.createOrReplaceTempView("atores_silver")
df_avaliacoes.createOrReplaceTempView("avaliacoes_silver")
df_episodios.createOrReplaceTempView("episodios_silver")
df_filmes.createOrReplaceTempView("filmes_silver")
df_generos.createOrReplaceTempView("generos_silver")
df_pagamentos.createOrReplaceTempView("pagamentos_silver")
df_planos.createOrReplaceTempView("planos_silver")
df_series.createOrReplaceTempView("series_silver")
df_usuarios.createOrReplaceTempView("usuarios_silver")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS gold COMMENT 'Schema para a camada Gold, com dados agregados e modelados para análise.'

In [ ]:
from pyspark.sql.functions import col, floor, datediff, current_date

# Adicionando a coluna calculada IDADE_ATUAL
df_dim_usuario = spark.table("usuarios_silver").withColumn(
    "IDADE_ATUAL",
    floor(datediff(current_date(), col("DATA_NASCIMENTO")) / 365.25)
)

# Persistindo a dimensão na camada Gold
df_dim_usuario.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_usuario")

# Criando view para uso futuro na criação da fato
df_dim_usuario.createOrReplaceTempView("dim_usuario")

print("Tabela gold.dim_usuario criada com sucesso.")
display(spark.table("gold.dim_usuario"))

In [ ]:
# Persistindo a dimensão na camada Gold
spark.table("planos_silver").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_plano")

# Criando view
spark.table("gold.dim_plano").createOrReplaceTempView("dim_plano")

print("Tabela gold.dim_plano criada com sucesso.")
display(spark.table("gold.dim_plano"))

In [ ]:
# Unificando filmes e episódios em uma única tabela
dim_conteudo_query = """
    -- Selecionando Filmes
    SELECT
        f.ID_FILME AS ID_CONTEUDO_ORIGEM,
        'Filme' AS TIPO_CONTEUDO,
        f.TITULO AS TITULO_CONTEUDO,
        NULL AS TITULO_SERIE,
        NULL AS TEMPORADA,
        f.DURACAO_MIN,
        g.NOME_GENERO,
        f.ANO_LANCAMENTO,
        f.CLASSIFICACAO
    FROM filmes_silver f
    LEFT JOIN generos_silver g ON f.ID_GENERO = g.ID_GENERO

    UNION ALL

    -- Selecionando Episódios de Séries
    SELECT
        e.ID_EPISODIO AS ID_CONTEUDO_ORIGEM,
        'Série' AS TIPO_CONTEUDO,
        e.TITULO AS TITULO_CONTEUDO,
        s.TITULO AS TITULO_SERIE,
        e.TEMPORADA,
        e.DURACAO_MIN,
        g.NOME_GENERO,
        s.ANO_ESTREIA AS ANO_LANCAMENTO,
        NULL AS CLASSIFICACAO
    FROM episodios_silver e
    LEFT JOIN series_silver s ON e.ID_SERIE = s.ID_SERIE
    LEFT JOIN generos_silver g ON s.ID_GENERO = g.ID_GENERO
"""

df_dim_conteudo = spark.sql(dim_conteudo_query)

# Persistindo a dimensão na camada Gold
df_dim_conteudo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_conteudo")

# Criando view
df_dim_conteudo.createOrReplaceTempView("dim_conteudo")

print("Tabela gold.dim_conteudo criada com sucesso.")
display(spark.table("gold.dim_conteudo"))

In [ ]:
from pyspark.sql.functions import sequence, to_date, explode, date_format, year, month, quarter, dayofmonth, dayofweek

# Gerando a dimensão de tempo de 2023 a 2025
df_dim_tempo = (
    spark.sql("SELECT explode(sequence(to_date('2023-01-01'), to_date('2025-12-31'), interval 1 day)) as DATA_COMPLETA")
    .withColumn("ANO", year("DATA_COMPLETA"))
    .withColumn("MES", month("DATA_COMPLETA"))
    .withColumn("NOME_MES", date_format("DATA_COMPLETA", "MMMM"))
    .withColumn("TRIMESTRE", quarter("DATA_COMPLETA"))
    .withColumn("DIA", dayofmonth("DATA_COMPLETA"))
    .withColumn("DIA_DA_SEMANA", dayofweek("DATA_COMPLETA"))
    .withColumn("NOME_DIA_DA_SEMANA", date_format("DATA_COMPLETA", "E"))
)

# Persistindo a dimensão na camada Gold
df_dim_tempo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.dim_tempo")

print("Tabela gold.dim_tempo criada com sucesso.")
display(spark.table("gold.dim_tempo"))

In [ ]:
fato_query = """
    -- Passo 1: Agregar Pagamentos por Mês/Usuário
    WITH pagamentos_mensal AS (
        SELECT
            ID_USUARIO,
            ID_PLANO,
            date_format(DATA_PAGAMENTO, 'yyyy-MM') AS MES_ANO,
            SUM(VALOR) as VALOR_PAGO_MES
        FROM pagamentos_silver
        GROUP BY ID_USUARIO, ID_PLANO, date_format(DATA_PAGAMENTO, 'yyyy-MM')
    ),
    
    -- Passo 2: Agregar Minutos Assistidos por Mês/Usuário
    assistencias_com_duracao AS (
        SELECT
            a.ID_USUARIO,
            date_format(a.DATA_ASSISTENCIA, 'yyyy-MM') AS MES_ANO,
            dc.DURACAO_MIN
        FROM assistencias_silver a
        INNER JOIN dim_conteudo dc ON a.ID_CONTEUDO = dc.ID_CONTEUDO_ORIGEM
    ),
    
    consumo_mensal AS (
        SELECT
            ID_USUARIO,
            MES_ANO,
            SUM(DURACAO_MIN) as MINUTOS_ASSISTIDOS_MES,
            COUNT(1) as QTDE_ASSISTENCIAS_MES
        FROM assistencias_com_duracao
        GROUP BY ID_USUARIO, MES_ANO
    ),
    
    -- Passo 3: Agregar Avaliações por Mês/Usuário
    avaliacoes_mensal AS (
        SELECT
            ID_USUARIO,
            date_format(DATA_AVALIACAO, 'yyyy-MM') AS MES_ANO,
            COUNT(ID_AVALIACAO) as QTDE_AVALIACOES_MES
        FROM avaliacoes_silver
        GROUP BY ID_USUARIO, MES_ANO
    ),
    
    -- Passo 4: Unir todas as métricas em uma única tabela
    base_unificada AS (
        SELECT
            COALESCE(p.ID_USUARIO, c.ID_USUARIO, a.ID_USUARIO) AS ID_USUARIO,
            COALESCE(p.MES_ANO, c.MES_ANO, a.MES_ANO) AS MES_ANO,
            p.ID_PLANO,
            p.VALOR_PAGO_MES,
            c.MINUTOS_ASSISTIDOS_MES,
            c.QTDE_ASSISTENCIAS_MES,
            a.QTDE_AVALIACOES_MES
        FROM pagamentos_mensal p
        FULL OUTER JOIN consumo_mensal c ON p.ID_USUARIO = c.ID_USUARIO AND p.MES_ANO = c.MES_ANO
        FULL OUTER JOIN avaliacoes_mensal a ON COALESCE(p.ID_USUARIO, c.ID_USUARIO) = a.ID_USUARIO AND COALESCE(p.MES_ANO, c.MES_ANO) = a.MES_ANO
    )
    
    -- Passo 5: Montar a fato final com as chaves FK e flags
    SELECT
        -- Chaves Estrangeiras (FKs)
        u.ID_USUARIO AS FK_USUARIO,
        p.ID_PLANO AS FK_PLANO,
        b.MES_ANO,
        
        -- Métricas
        COALESCE(b.VALOR_PAGO_MES, 0) as VALOR_PAGO_MES,
        COALESCE(b.MINUTOS_ASSISTIDOS_MES, 0) as MINUTOS_ASSISTIDOS_MES,
        COALESCE(b.QTDE_ASSISTENCIAS_MES, 0) as QTDE_ASSISTENCIAS_MES,
        COALESCE(b.QTDE_AVALIACOES_MES, 0) as QTDE_AVALIACOES_MES,
        
        -- Flag de Atividade
        CASE WHEN COALESCE(b.VALOR_PAGO_MES, 0) > 0 OR COALESCE(b.MINUTOS_ASSISTIDOS_MES, 0) > 0 THEN 1 ELSE 0 END AS FLAG_ATIVO_MES
        
    FROM base_unificada b
    LEFT JOIN dim_usuario u ON b.ID_USUARIO = u.ID_USUARIO
    LEFT JOIN dim_plano p ON b.ID_PLANO = p.ID_PLANO
"""

df_fato_mensal = spark.sql(fato_query)

# Persistindo a fato na camada Gold
df_fato_mensal.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fato_resumo_usuario_mensal")

print("Tabela gold.fato_resumo_usuario_mensal criada com sucesso.")
display(spark.table("gold.fato_resumo_usuario_mensal"))

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW metrica_minutos_assistidos AS
SELECT
  MES_ANO,
  SUM(MINUTOS_ASSISTIDOS_MES) as TOTAL_MINUTOS_ASSISTIDOS
FROM gold.fato_resumo_usuario_mensal
GROUP BY MES_ANO
ORDER BY MES_ANO;

SELECT * FROM metrica_minutos_assistidos;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW metrica_novos_usuarios AS
SELECT
  date_format(DATA_CADASTRO, 'yyyy-MM') as MES_CADASTRO,
  COUNT(ID_USUARIO) as NOVOS_USUARIOS
FROM gold.dim_usuario
GROUP BY date_format(DATA_CADASTRO, 'yyyy-MM')
ORDER BY MES_CADASTRO;

SELECT * FROM metrica_novos_usuarios;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW kpi_mrr AS
SELECT
  MES_ANO,
  SUM(VALOR_PAGO_MES) as MRR
FROM gold.fato_resumo_usuario_mensal
GROUP BY MES_ANO
ORDER BY MES_ANO;

SELECT * FROM kpi_mrr;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW kpi_churn_rate AS
WITH atividade_usuario_mes AS (
  SELECT DISTINCT
    FK_USUARIO,
    MES_ANO,
    FLAG_ATIVO_MES
  FROM gold.fato_resumo_usuario_mensal
),
atividade_com_mes_anterior AS (
  SELECT
    FK_USUARIO,
    MES_ANO,
    FLAG_ATIVO_MES,
    LAG(FLAG_ATIVO_MES, 1, 0) OVER (PARTITION BY FK_USUARIO ORDER BY MES_ANO) as ATIVO_MES_ANTERIOR
  FROM atividade_usuario_mes
),
churn_events AS (
  SELECT
    MES_ANO,
    SUM(ATIVO_MES_ANTERIOR) as TOTAL_ATIVOS_MES_ANTERIOR,
    SUM(CASE WHEN ATIVO_MES_ANTERIOR = 1 AND FLAG_ATIVO_MES = 0 THEN 1 ELSE 0 END) as CHURNED_USERS
  FROM atividade_com_mes_anterior
  GROUP BY MES_ANO
)
SELECT
  MES_ANO,
  (CHURNED_USERS / TOTAL_ATIVOS_MES_ANTERIOR) * 100 as TAXA_DE_CHURN_PERCENTUAL
FROM churn_events
WHERE TOTAL_ATIVOS_MES_ANTERIOR > 0
ORDER BY MES_ANO;

SELECT * FROM kpi_churn_rate;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW kpi_ltv AS
WITH arpu AS ( -- Average Revenue Per User
  SELECT
    k.MES_ANO,
    k.MRR / a.TOTAL_ATIVOS_MES_ANTERIOR AS ARPU_MENSAL
  FROM kpi_mrr k
  JOIN (
    SELECT MES_ANO, SUM(ATIVO_MES_ANTERIOR) as TOTAL_ATIVOS_MES_ANTERIOR
    FROM (
      SELECT
        MES_ANO,
        LAG(FLAG_ATIVO_MES, 1, 0) OVER (PARTITION BY FK_USUARIO ORDER BY MES_ANO) as ATIVO_MES_ANTERIOR
      FROM gold.fato_resumo_usuario_mensal
    )
    GROUP BY MES_ANO
  ) a ON k.MES_ANO = a.MES_ANO
  WHERE a.TOTAL_ATIVOS_MES_ANTERIOR > 0
),
churn AS (
  SELECT
    MES_ANO,
    TAXA_DE_CHURN_PERCENTUAL / 100 as CHURN_RATE
  FROM kpi_churn_rate
)
SELECT
  a.MES_ANO,
  a.ARPU_MENSAL / c.CHURN_RATE as LTV_ESTIMADO
FROM arpu a
JOIN churn c ON a.MES_ANO = c.MES_ANO
WHERE c.CHURN_RATE > 0
ORDER BY a.MES_ANO;

SELECT * FROM kpi_ltv;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW kpi_engajamento_pais_plano AS
SELECT
  f.MES_ANO,
  u.PAIS,
  p.NOME_PLANO,
  AVG(f.MINUTOS_ASSISTIDOS_MES) as MEDIA_MINUTOS_ASSISTIDOS
FROM gold.fato_resumo_usuario_mensal f
JOIN gold.dim_usuario u ON f.FK_USUARIO = u.ID_USUARIO
JOIN gold.dim_plano p ON f.FK_PLANO = p.ID_PLANO
WHERE f.FLAG_ATIVO_MES = 1
GROUP BY f.MES_ANO, u.PAIS, p.NOME_PLANO
ORDER BY f.MES_ANO, u.PAIS, MEDIA_MINUTOS_ASSISTIDOS DESC;

SELECT * FROM kpi_engajamento_pais_plano;